# RL-Crystal-Gen Tutorial: Ag-Bi-I Structure Generation with E_hull Reward

This notebook demonstrates the full workflow:
1. Set up reward components (E_hull, validity, composition, diversity)
2. Initialize the MockGenerator (or Chemeleon2 LDM)
3. Run GRPO reinforcement learning
4. Analyze generated structures
5. Add a custom reward

**No GPU required** — the notebook uses MockGenerator for demonstration.

In [ ]:
import sys
sys.path.insert(0, '..')  # add repo root to path

import torch
import matplotlib.pyplot as plt
from pymatgen.core import Structure

## 1. Build the Composite Reward

In [ ]:
from src.rewards import (
    CompositeReward,
    EhullReward,
    ValidityReward,
    CompositionReward,
    DiversityReward,
)
from src.rl.reward_builder import build_ag_bi_i_ehull_reward

# Option A: Pre-built preset
reward_fn = build_ag_bi_i_ehull_reward(
    ehull_backend='mace',   # 'mp_api' requires MP_API_KEY env var
    ehull_weight=1.0,
    validity_weight=0.5,
    composition_weight=0.5,
    diversity_weight=0.3,
)

print(reward_fn.summary())

In [ ]:
# Option B: Build manually for full control
reward_fn_custom = CompositeReward(components=[
    EhullReward(
        backend='mace',
        weight=1.0,
        normalize='zscore',
        stable_threshold=0.1,   # eV/atom
        nan_penalty=-2.0,
    ),
    ValidityReward(
        weight=0.5,
        normalize='none',
        check_charge=True,
        check_min_dist=True,
    ),
    CompositionReward(
        required_elements=['Ag', 'Bi', 'I'],
        weight=0.5,
        normalize='none',
    ),
    DiversityReward(
        weight=0.3,
        normalize='zscore',
    ),
])
print(reward_fn_custom.summary())

## 2. Initialize the Generator

In [ ]:
from src.generator.chemeleon2_wrapper import MockGenerator

# MockGenerator: random Ag-Bi-I structures, no GPU needed
generator = MockGenerator(seed=42)

# Test: generate 5 structures
test_structures = generator.sample(n=5)
for i, s in enumerate(test_structures):
    print(f'  [{i}] {s.composition.reduced_formula:12s}  '
          f'density={s.density:.2f} g/cm³  '
          f'n_sites={s.num_sites}')

In [ ]:
# To use Chemeleon2 LDM instead (requires checkpoints):
# from src.generator.chemeleon2_wrapper import Chemeleon2Generator
# generator = Chemeleon2Generator(
#     checkpoint_path='checkpoints/ldm_best.ckpt',
#     vae_checkpoint_path='checkpoints/vae_best.ckpt',
#     device='cuda',
# )

## 3. Compute Rewards on a Test Batch

In [ ]:
# Generate a batch and compute rewards
structures = generator.sample(n=16)

# Use validity + composition only (E_hull requires API or MACE)
quick_reward = CompositeReward(components=[
    ValidityReward(weight=1.0, normalize='none'),
    CompositionReward(required_elements=['Ag', 'Bi', 'I'], weight=1.0, normalize='none'),
    DiversityReward(weight=0.5, normalize='minmax'),
])

rewards = quick_reward(structures)
print(f'Rewards shape: {rewards.shape}')
print(f'Mean reward:   {rewards.mean():.4f}')
print(f'Max reward:    {rewards.max():.4f}')
print(f'Min reward:    {rewards.min():.4f}')

## 4. Run GRPO Training

In [ ]:
from src.rl.grpo_trainer import GRPOTrainer, GRPOConfig

config = GRPOConfig(
    n_iterations=20,          # short demo; use 500+ for real training
    batch_size=16,
    group_size=4,
    lr=1e-5,
    output_dir='outputs/tutorial_run',
    log_every=5,
    save_every=10,
    seed=42,
)

trainer = GRPOTrainer(
    generator=generator,
    reward_fn=quick_reward,
    config=config,
    condition={'elements': ['Ag', 'Bi', 'I']},
)

history = trainer.train()

In [ ]:
# Plot reward curve
iterations = [h['iteration'] for h in history]
means = [h['reward_mean'] for h in history]
stds  = [h['reward_std'] for h in history]

plt.figure(figsize=(8, 4))
plt.plot(iterations, means, 'b-o', markersize=4, label='Mean reward')
plt.fill_between(
    iterations,
    [m - s for m, s in zip(means, stds)],
    [m + s for m, s in zip(means, stds)],
    alpha=0.2, color='b', label='±1 std'
)
plt.xlabel('RL Iteration')
plt.ylabel('Composite Reward')
plt.title('Ag-Bi-I Structure Generation: Reward Curve')
plt.legend()
plt.tight_layout()
plt.show()

## 5. Evaluate Final Structures

In [ ]:
eval_result = trainer.evaluate(n=50)
print(f"Evaluation over {eval_result['n_structures']} structures:")
print(f"  Reward mean: {eval_result['reward_mean']:.4f}")
print(f"  Reward std:  {eval_result['reward_std']:.4f}")

# Show top 5 structures by reward
structures = eval_result['structures']
rewards = eval_result['rewards']
ranked = sorted(zip(rewards.tolist(), structures), reverse=True)

print('\nTop 5 structures:')
for r, s in ranked[:5]:
    print(f'  reward={r:.4f}  formula={s.composition.reduced_formula:12s}  '
          f'density={s.density:.2f}')

## 6. Add a Custom Reward (Band Gap)

In [ ]:
# Load the custom band gap reward from custom_rewards/
from custom_rewards.bandgap_reward import BandGapReward

# Augment the reward function
extended_reward = CompositeReward(components=[
    ValidityReward(weight=1.0, normalize='none'),
    CompositionReward(required_elements=['Ag', 'Bi', 'I'], weight=0.5, normalize='none'),
    DiversityReward(weight=0.3, normalize='minmax'),
    BandGapReward(
        target_min=1.0,   # target band gap range: 1.0 – 2.0 eV (solar cell window)
        target_max=2.0,
        use_heuristic=True,
        weight=0.8,
        normalize='zscore',
    ),
])

print(extended_reward.summary())

# Test on a batch
test_structs = generator.sample(n=8)
test_rewards = extended_reward(test_structs)
print(f'\nExtended rewards: {test_rewards.numpy().round(3)}')

## 7. Next Steps

- Replace `MockGenerator` with `Chemeleon2Generator` using pretrained checkpoints
- Set `MP_API_KEY` and use `EhullReward(backend='mp_api')` for accurate E_hull
- Add `EhullReward(backend='mace')` for fast GPU-based estimation
- Write your own reward in `custom_rewards/` and add it to the YAML config
- Run longer training: `n_iterations=500`, `batch_size=64` on a GPU node
- Enable W&B logging with `--wandb_project my-project` in `scripts/train_rl.py`